In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io
from helper import *

from sklearn.model_selection import GridSearchCV

In [23]:
import warnings
warnings.filterwarnings('ignore')

In [24]:
import mlflow
import mlflow.pytorch

In [25]:
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

In [26]:
mlflow.set_experiment("Clasificador_Imagenes_CNN")

<Experiment: artifact_location=('file:///c:/Users/Camila/OneDrive/Escritorio/Redes '
 'Neuronales/Skin-dataset-classification-CS2026/mlruns/7'), creation_time=1782143417987, experiment_id='7', last_update_time=1782143417987, lifecycle_stage='active', name='Clasificador_Imagenes_CNN', tags={}, trace_location=None, workspace='default'>

In [27]:
def log_classification_report(model, loader, writer, device, classes, step, prefix="val"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig_cm, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{prefix.title()} - Confusion Matrix')

    # Guardar localmente y subir a MLflow
    fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
    fig_cm.savefig(fig_path)
    mlflow.log_artifact(fig_path)
    os.remove(fig_path)

    plot_to_tensorboard(fig_cm, writer, f"{prefix}/confusion_matrix", step)

    cls_report = classification_report(all_labels, all_preds, target_names=classes)
    writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    # También loguear texto del reporte
    with open(f"classification_report_{prefix}_epoch_{step}.txt", "w") as f:
        f.write(cls_report)
    mlflow.log_artifact(f.name)
    os.remove(f.name)


In [28]:
# Entrenamiento y validación
def evaluate(model, loader, writer, device, classes, epoch=None, prefix="val"):
    log_classification_report(model, loader, writer, device, classes, step=epoch , prefix="val")
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            # Loguear imágenes del primer batch
            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)

    return avg_loss, acc

In [29]:
# Paths
train_dir = "data/Split_smol/train"
val_dir = "data/Split_smol/val/"

In [30]:
# Crear directorio de logs de tensorboard
log_dir = "runs/experimento_cnn"
writer = SummaryWriter(log_dir=log_dir)

In [31]:
np.random.rand()

0.29383745204890244

In [39]:
hparams_space = {
    "model": "CNNClassifier",
    "input_size": [64],
    "batch_size": [64, 128],
    "lr": [1e-4, 3e-4, 5e-4, 1e-3],
    "epochs": 120,
    "optimizer": ["Adam"],
    "HFlip": [0.0, 0.5],
    "VFlip": [0.0],
    "RBContrast": [0.0, 0.3, 0.5],
    "weight_decay": [0.0, 1e-5, 1e-4],
    "loss_fn": "CrossEntropyLoss",
    "train_dir": train_dir,
    "val_dir": val_dir,
    "es_patience": 12,
    "dropout": [0.2, 0.3, 0.4],
}

In [40]:
modelnbr = 0
for input_size in hparams_space["input_size"]:
    for batch_size in hparams_space["batch_size"]:
        for lr in hparams_space["lr"]:
            for optimizer in hparams_space["optimizer"]:
                for HFlip in hparams_space["HFlip"]:
                    for VFlip in hparams_space["VFlip"]:
                        for RBContrast in hparams_space["RBContrast"]:
                            for dropout in hparams_space["dropout"]:
                                if np.random.rand() < 0.05:
                                    print(f"modelo número: {modelnbr}", end = "\r")
                                    modelnbr += 1
                                    hparams= {
                                        "model": ("CNNClassifier"),
                                        "input_size":  input_size,
                                        "batch_size": batch_size,
                                        "lr": lr,
                                        "epochs": 50,
                                        "optimizer": optimizer,
                                        "HFlip": HFlip,
                                        "VFlip": VFlip,
                                        "RBContrast": RBContrast,
                                        "loss_fn": "CrossEntropyLoss",
                                        "train_dir": train_dir,
                                        "val_dir": val_dir,
                                        "es_patience": 5,
                                        "dropout": dropout,
                                    }
                                    train_transform = A.Compose([
                                        A.Resize(hparams["input_size"], hparams["input_size"]),
                                        A.HorizontalFlip(p=hparams["HFlip"]),
                                        A.VerticalFlip(p=hparams["VFlip"]),
                                        A.RandomBrightnessContrast(p=hparams["RBContrast"]),
                                        A.Normalize(),
                                        ToTensorV2()
                                    ])
                                    val_test_transform = A.Compose([
                                        A.Resize(hparams["input_size"], hparams["input_size"]),
                                        A.Normalize(),
                                        ToTensorV2()
                                    ])
                                    train_dataset = CustomImageDataset(train_dir, transform=train_transform)
                                    val_dataset   = CustomImageDataset(val_dir, transform=val_test_transform)
                                    batch_size = hparams["batch_size"]
                                    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
                                    val_loader   = DataLoader(val_dataset, batch_size=batch_size)
                                    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
                                    num_classes = len(set(train_dataset.labels))
                                    model = CNNClassifier(num_classes=num_classes, input_size = hparams["input_size"], dropout = hparams["dropout"]).to(device)
                                    criterion = nn.CrossEntropyLoss()
                                    optimizer = optim.Adam(model.parameters(), lr=hparams["lr"]) if hparams["optimizer"]=="Adam" else optim.SGD(model.parameters(), lr=hparams["lr"])
                                    hparams["count_params"] = count_parameters(model)
                                    with mlflow.start_run():
                                        # Log hiperparámetros
                                        mlflow.log_params(hparams)
                                        best_val_acc = 0
                                        best_val_loss = 0
                                        best_train_acc = 0
                                        best_train_loss = 0
                                        best_epoch = 0
                                        for epoch in range(hparams["epochs"]):
                                            model.train()
                                            running_loss = 0.0
                                            correct, total = 0, 0
                                        
                                            for images, labels in train_loader:
                                                images, labels = images.to(device), labels.to(device)
                                        
                                                optimizer.zero_grad()
                                                outputs = model(images)
                                                loss = criterion(outputs, labels)
                                                loss.backward()
                                                optimizer.step()
                                        
                                                running_loss += loss.item()
                                                _, preds = torch.max(outputs, 1)
                                                correct += (preds == labels).sum().item()
                                                total += labels.size(0)
                                        
                                            train_loss = running_loss / len(train_loader)
                                            train_acc = 100.0 * correct / total
                                            val_loss, val_acc = evaluate(model, val_loader, writer, device,train_dataset.label_encoder.classes_,epoch=epoch, prefix="val")
                                        
                                            print(f"Epoch {epoch+1}:")
                                            print(f"  Train Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%")
                                            print(f"  Val   Loss: {val_loss:.4f}, Accuracy: {val_acc:.2f}%")
                                        
                                            writer.add_scalar("train/loss", train_loss, epoch)
                                            writer.add_scalar("train/accuracy", train_acc, epoch)
                                        
                                            # Log en MLflow
                                            mlflow.log_metrics({
                                                "train_loss": train_loss,
                                                "train_accuracy": train_acc,
                                                "val_loss": val_loss,
                                                "val_accuracy": val_acc
                                            }, step=epoch)
                                            if val_acc > best_val_acc:
                                                best_val_acc = val_acc
                                                best_val_loss = val_loss
                                                best_train_acc = train_acc
                                                best_train_loss = train_loss
                                                best_epoch = epoch
                                                # Guardar modelo
                                                torch.save(model.state_dict(), "mlp_model.pth")
                                                print("Modelo guardado como 'mlp_model.pth'")
                                                mlflow.log_artifact("mlp_model.pth")
                                                #mlflow.pytorch.log_model(model, artifact_path="pytorch_model")
                                            elif epoch > best_epoch + hparams["es_patience"]:
                                                print("Early Stopping")
                                                break
                                                
                                        mlflow.log_metrics({
                                                "train_loss": best_train_loss,
                                                "train_accuracy": best_train_acc,
                                                "val_loss": best_val_loss,
                                                "val_accuracy": best_val_acc,
                                                "best_epoch": best_epoch
                                            }, step=epoch+1)                                                
                                        

Epoch 1:úmero: 0
  Train Loss: 2.1054, Accuracy: 18.65%
  Val   Loss: 2.0956, Accuracy: 34.44%
Modelo guardado como 'mlp_model.pth'
Epoch 2:
  Train Loss: 1.8868, Accuracy: 32.63%
  Val   Loss: 2.0169, Accuracy: 33.33%
Epoch 3:
  Train Loss: 1.7248, Accuracy: 43.16%
  Val   Loss: 1.9414, Accuracy: 37.22%
Modelo guardado como 'mlp_model.pth'
Epoch 4:
  Train Loss: 1.6122, Accuracy: 43.01%
  Val   Loss: 1.8715, Accuracy: 35.00%
Epoch 5:
  Train Loss: 1.5316, Accuracy: 44.51%
  Val   Loss: 1.8203, Accuracy: 38.33%
Modelo guardado como 'mlp_model.pth'
Epoch 6:
  Train Loss: 1.4571, Accuracy: 46.92%
  Val   Loss: 1.7717, Accuracy: 41.11%
Modelo guardado como 'mlp_model.pth'
Epoch 7:
  Train Loss: 1.3937, Accuracy: 50.68%
  Val   Loss: 1.7536, Accuracy: 36.67%
Epoch 8:
  Train Loss: 1.3680, Accuracy: 49.62%
  Val   Loss: 1.7062, Accuracy: 45.56%
Modelo guardado como 'mlp_model.pth'
Epoch 9:
  Train Loss: 1.2691, Accuracy: 54.44%
  Val   Loss: 1.6683, Accuracy: 44.44%
Epoch 10:
  Train Loss: 

In [41]:
import mlflow
import pandas as pd

experiment = mlflow.get_experiment_by_name("Clasificador_Imagenes_CNN")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.val_accuracy DESC"]
)

cols = [
    "metrics.val_accuracy",
    "metrics.val_loss",
    "metrics.train_accuracy",
    "metrics.train_loss",
    "params.input_size",
    "params.batch_size",
    "params.lr",
    "params.dropout",
    "params.HFlip",
    "params.VFlip",
    "params.RBContrast",
    "params.optimizer",
    "params.es_patience"
]

top_runs = runs[cols].sort_values("metrics.val_accuracy", ascending=False)
top_runs.head(10)

,metrics.val_accuracy,metrics.val_loss,metrics.train_accuracy,metrics.train_loss,params.input_size,params.batch_size,params.lr,params.dropout,params.HFlip,params.VFlip,params.RBContrast,params.optimizer,params.es_patience
0,61.111111,1.181201,83.759398,0.449320,64,128,0.001,0.4,0.0,0.0,0.5,Adam,5
1,61.111111,1.309393,98.345865,0.087152,64,64,0.001,0.4,0.0,0.0,0.0,Adam,5
2,61.111111,1.245295,81.052632,0.619490,64,128,0.0005,0.3,0.0,0.0,0.5,Adam,5
3,61.111111,1.207556,82.255639,0.558956,64,64,0.0001,0.3,0.0,0.0,0.0,Adam,5
4,60.555556,1.160845,84.210526,0.496142,64,128,0.0005,0.2,0.0,0.0,0.3,Adam,5
5,60.000000,1.224089,79.699248,0.534784,64,128,0.001,0.2,0.0,0.0,0.5,Adam,5
6,60.000000,1.254119,76.090226,0.651400,128,128,0.0001,0.2,0.0,0.0,0.3,Adam,5
7,59.444444,1.248655,70.375940,0.792694,64,128,0.0001,0.3,0.5,0.0,0.0,Adam,5
8,59.444444,1.150277,83.458647,0.488062,128,32,0.0005,0.3,0.0,0.0,0.3,Adam,5
9,59.444444,1.467306,95.488722,0.145881,64,64,0.001,0.3,0.0,0.0,0.0,Adam,5


In [42]:
best_hparams = {
    "input_size": 64,
    "batch_size": 128,
    "lr": 5e-4,
    "dropout": 0.3,
    "HFlip": 0.0,
    "VFlip": 0.0,
    "RBContrast": 0.5,
}

In [46]:
test_transform = A.Compose([
    A.Resize(best_hparams["input_size"], best_hparams["input_size"]),
    A.Normalize(),
    ToTensorV2()
])

In [47]:
test_dataset = CustomImageDataset(test_dir, transform=test_transform)

test_loader = DataLoader(
    test_dataset,
    batch_size=best_hparams["batch_size"],
    shuffle=False
)

NameError: name 'test_dir' is not defined